In [2]:
# Run this cell at the beginning of each session to set up the environment
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/cs372_final_project')

# First time: clone repo
# !git clone https://github.com/victoriaf55/cs372-final.git

# Every subsequent session: pull latest
!git -C cs372-final pull

# Install dependencies
!apt-get install -y xvfb ffmpeg > /dev/null 2>&1
!pip install gymnasium[mujoco] torch tensorboard pyvirtualdisplay > /dev/null 2>&1

# Verify installation
import gymnasium as gym

env = gym.make("InvertedPendulum-v5")
obs, info = env.reset()
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
env.close()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.38 KiB | 16.00 KiB/s, done.
From https://github.com/victoriaf55/cs372-final
   87a828c..d10fe7c  main       -> origin/main
Updating 87a828c..d10fe7c
Fast-forward
 ATTRIBUTION.md        |  6 ++--
 evaluate.py           | 21 --------------
 notebooks/train.ipynb | 76 ++++++++++++++++++++++++++++++++++++++++++---------
 3 files changed, 66 insertions(+), 37 deletions(-)
Observation space: Box(-inf, inf, (4,), float64)
Action space: Box(-3.0, 3.0, (1,), float32)


In [4]:
# =============================================================================
# AI-GENERATED CODE
# Generated by: Claude (Anthropic), claude-sonnet-4-6
# Date: April 2026
# Prompt context: Random baseline evaluation for InvertedPendulum-v5
# Modified by: Victoria Feng
# =============================================================================

import numpy as np

env = gym.make("InvertedPendulum-v5")
rewards = []

for episode in range(100):
    obs, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
    rewards.append(total_reward)

print(f"Mean Reward: {np.mean(rewards):.2f}")
print(f"Std Reward: {np.std(rewards):.2f}")
print(f"Min Reward: {np.min(rewards):.2f}")
print(f"Max Reward: {np.max(rewards):.2f}")
print(f"Median Reward: {np.median(rewards):.2f}")
print(f"Mean Ep Length: {len(rewards) / 100:.2f}")
print(f"Success Rate: {np.mean([r==1000 for r in rewards]) * 100:.2f}%")
env.close()

Mean Reward: 4.59
Std Reward: 3.13
Min Reward: 2.00
Max Reward: 24.00
Median Reward: 3.50
Mean Ep Length: 1.00
Success Rate: 0.00%


In [ ]:
# Save random baseline model (untrained weights)
import torch
import gymnasium as gym
import sys
import os

sys.path.append('/content/drive/MyDrive/cs372_final_project/cs372-final')

from models.actor_critic import ActorCritic

HIDDEN_DIM = 64
CHECKPOINT_PATH = "/content/drive/MyDrive/cs372_final_project/checkpoints"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

env = gym.make("InvertedPendulum-v5")
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
env.close()

random_model = ActorCritic(obs_dim, act_dim, HIDDEN_DIM)
torch.save(
    random_model.state_dict(),
    f"{CHECKPOINT_PATH}/random_baseline.pt"
)
print("Random baseline model saved.")

!python evaluate.py --checkpoint /content/drive/MyDrive/cs372_project/checkpoints/random_baseline.pt --record

Random baseline
Episodes: 100
Mean Reward: 4.59
Std Reward: 3.13
Min Reward: 2.00
Max Reward: 24.00
Median Reward: 3.50
Mean Ep Length: 1.00
Success Rate: 0.00%

In [34]:
# Run training loop
import os
os.chdir('/content/drive/MyDrive/cs372_final_project/cs372-final')
!python train.py

2026-04-25 02:43:09.367888: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777084989.407137   84780 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777084989.420701   84780 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777084989.453173   84780 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777084989.453203   84780 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777084989.453213   84780 computation_placer.cc:177] computation placer alr

In [39]:
import os
os.chdir('/content/drive/MyDrive/cs372_final_project/cs372-final')

# Evaluate latest checkpoint
# !python evaluate.py

# Evaluate specific checkpoint
!python evaluate.py --checkpoint /content/drive/MyDrive/cs372_final_project/checkpoints/ppo_update_350.pt

# Evaluate and record video
# !python evaluate.py --record

# Compare all checkpoints
# !python evaluate.py --compare

# Change number of episodes
# !python evaluate.py --n_episodes 50

Loaded checkpoint: /content/drive/MyDrive/cs372_final_project/checkpoints/ppo_update_350.pt

EVALUATION RESULTS
Episodes:       100
Mean Reward:    925.73
Std Reward:     203.83
Min Reward:     74.00
Max Reward:     1000.00
Median Reward:  1000.00
Mean Ep Length: 925.88
Success Rate:   85.0%

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/cs372_final_project/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
Video saved to /content/drive/MyDrive/cs372_final_project/videos | Episode reward: 1000.00


: 

: 

==================================================
EVALUATION RESULTS - Update 350, Timestep 1433600
==================================================
Episodes:       100
Mean Reward:    925.73
Std Reward:     203.83
Min Reward:     74.00
Max Reward:     1000.00
Median Reward:  1000.00
Mean Ep Length: 925.88
Success Rate:   85.0%
==================================================